# Assignment 8: Classifying High Exam Scores
## Part 1 — Define a Classification Problem

In the previous assignment, exam_score was a number. Here we turn that number into a two-class label:
- high_exam_score = Yes if exam_score >= 80
- high_exam_score = No if exam_score < 80

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab. Using class1_student_habits.csv from this folder.")

In [ ]:
import pandas as pd

# "None" is a real answer for prior programming experience.
# The default reader would treat that word as a missing value.
df = pd.read_csv(
    "class1_student_habits.csv",
    keep_default_na=False,
    na_values=[""],
)

df["high_exam_score"] = "No"
df.loc[df["exam_score"] >= 80, "high_exam_score"] = "Yes"

df[["student_id", "exam_score", "high_exam_score"]].head(10)

The new column is a label, not a score. A student with 79.9 is No, and a student with 80.0 is Yes. 

In [ ]:
class_counts = df["high_exam_score"].value_counts()
class_percentages = df["high_exam_score"].value_counts(normalize=True) * 100

print("Number of students:", len(df))
print()
print("Students in each class:")
print(class_counts)
print()
print("Percentage of students in each class:")
print(class_percentages.round(1))

### Class distribution

In [ ]:
import matplotlib.pyplot as plt

order = ["No", "Yes"]
counts = df["high_exam_score"].value_counts().reindex(order)

plt.figure()
plt.bar(order, counts, edgecolor="black")
plt.title("Class Distribution of high_exam_score")
plt.xlabel("high_exam_score")
plt.ylabel("Number of students")
plt.show()

## Questions for Part 1

Why is predicting exam_score a regression problem? exam_score is a number on a continuous scale, from the low 50s up to 100 in this class. A regression model tries to predict that number. The size of the error matters, which is what regression measures.

Why is predicting high_exam_score a classification problem? high_exam_score is a category with two possible answers: Yes or No. The model is not asked how many points the student earned. It is asked which group the student belongs to. A prediction is either the right label or the wrong label. A score of 81 and a score of 100 are both Yes, and a score of 79 and a score of 52 are both No.

How many students belong to each class? Of the 250 students, 99 are Yes (exam_score of 80 or higher) and 151 are No (exam_score below 80).

What percentage of students belong to each class? Yes is 39.6% and No is 60.4%.

## Part 2 — Choose the Features

The primary model predicts high_exam_score from study habits and background. It uses these columns:

- study_hours_per_week
- sleep_hours_per_night
- attendance_rate
- practice_problems_per_week
- prior_programming_experience
- study_group
- commute_minutes

In [ ]:
feature_columns = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "prior_programming_experience",
    "study_group",
    "commute_minutes",
]

target_column = "high_exam_score"

X = df[feature_columns]
y = df[target_column]

print("Predictors:")
print(list(X.columns))
print()
print("Target:", target_column)
print()
print("Left out of the model:")
print(["student_id", "exam_score", "quiz_average"])
print()
print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

## Questions for Part 2

Why should student_id normally not be used as a predictor? student_id is only a label that tells the rows apart, such as S001 or S012. It is not a habit, and it does not describe how the student prepared. Each id belongs to one student, so a model that used it would be memorizing "this particular student scored high" instead of learning a pattern that could apply to someone new. A new student would have an id the model has never seen, so the id cannot help a real prediction.

Why would using exam_score to predict high_exam_score be a serious problem? high_exam_score was built directly from exam_score (Yes if the score is 80 or higher, No otherwise). If the model can see the exam score, it does not need study hours, sleep, or attendance at all, instead it can apply the same cutoff and get the label right every time. That accuracy would be meaningless, because the feature is the answer written in another form. This is target leakage. For a student who has not taken the exam yet, the score would not be available, which is exactly why the habits are the predictors and the exam score is not.

## Part 3 — Split the Data
The model should learn from most of the students and be checked on students it did not see. We hold out 20% as the test set and use the other 80% for training, with random_state=42 so the split stays the same each time the notebook is run.

The classes are uneven, about 60% No and 40% Yes, and the test set is only 50 students. The next cell compares a plain random split with a stratified split. We keep the stratified one.

In [ ]:
from sklearn.model_selection import train_test_split

def class_mix(labels):
    counts = labels.value_counts()
    percentages = labels.value_counts(normalize=True) * 100
    summary = pd.DataFrame({"count": counts, "percent": percentages.round(1)})
    return summary.reindex(["No", "Yes"])

X_train_plain, X_test_plain, y_train_plain, y_test_plain = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Full class:")
print(class_mix(y))
print()
print("Without stratification")
print("Training set:")
print(class_mix(y_train_plain))
print("Test set:")
print(class_mix(y_test_plain))
print()
print("With stratification")
print("Training set:")
print(class_mix(y_train))
print("Test set:")
print(class_mix(y_test))
print()
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## Questions for Part 3

What is the purpose of the training set? The training set is the data the model learns from. It sees each student's habits together with the known Yes or No label, and it adjusts so those habits line up with the label. In this split that is 200 students, 80% of the class.

What is the purpose of the test set? The test set is held back until after training. We use it to see how often the model is right on students it did not learn from. That is the check for how it might do on a new student. Here that is 50 students, or 20% of the class.

What does stratification do? It keeps the same class mix in the training set and the test set. In the full class, 60.4% of students are No and 39.6% are Yes. A plain split with random_state=42 does not protect that mix, since the test set came out 37 No and 13 Yes, which is 74% No and 26% Yes. The stratified split keeps both sets near the original mix. The training set is 121 No and 79 Yes (60.5% and 39.5%), and the test set is 30 No and 20 Yes (60% and 40%). We should stratify since the classes are uneven, and 50 test students is a small group, so a random split can make the test set look like a different class than the one the model trained on.

Why shouldn't we train the model using the test data? The test set only works as a check if the model has not already seen those students. If we train on them too, a high score can mean the model memorized those 50 rows instead of learning a pattern that works for someone new. We train on the 200 and score the model on the 50.

## Part 4 — Establish a Baseline

The next cell finds the most common class in the training data, predicts that class for every test student, and calculates the accuracy.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

majority_class = y_train.value_counts().idxmax()
majority_count = int(y_train.value_counts().max())

print("Training class counts:")
print(y_train.value_counts())
print()
print("Majority class:", majority_class)
print("Training students in that class:", majority_count, "of", len(y_train))

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

correct = int((baseline_predictions == y_test).sum())
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print()
print("Baseline prediction for every test student:", majority_class)
print("Test students predicted correctly:", correct, "of", len(y_test))
print("Baseline accuracy:", round(baseline_accuracy, 3))

## Questions for Part 4

What is the majority class? No. In the training set, 121 of 200 students are No and only 79 are Yes. The simple classifier ignores every habit and predicts No for every test student.

What accuracy does the majority-class baseline achieve? 60%. The test set has 30 students whose true label is No and 20 whose label is Yes. Predicting No for all 50 is right 30 times, and 30 / 50 = 0.60.

Why is this baseline useful? It is the accuracy you get without learning anything about study hours, sleep, attendance, or the other features. A later model is only doing useful work if it beats 60% on this same test set. An accuracy around 60% can look acceptable on its own and still be no better than always saying No. Because more students are No than Yes, a high accuracy does not by itself mean the model understands who will score 80 or above.

## Part 5 — Preprocess the Data

The pipeline below fills those blanks, rescales the numbers, and turns each category into 0/1 columns. It learns the fills and the scaling from the training set only, then applies those same steps to the test set. 

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "commute_minutes",
]
categorical_features = [
    "prior_programming_experience",
    "study_group",
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Missing values in the selected features:")
print(X[numeric_features + categorical_features].isna().sum())
print()
print("Prepared training shape:", X_train_prepared.shape)
print("Prepared test shape:", X_test_prepared.shape)
print()
print("Columns after preprocessing:")
print(list(preprocessor.get_feature_names_out()))

## Questions for Part 5

Why do categorical variables need to be converted before Logistic Regression can use them? Logistic Regression multiplies each feature by a weight and adds those products. It can only do that with numbers. prior_programming_experience contains the words None, Some, and Strong, and study_group contains Yes and No. Those words have to become numbers before the model can use them.

What does one-hot encoding do? It creates a separate 0/1 column for each category. study_group becomes two columns, one for No and one for Yes. A student in a study group gets 1 in the Yes column and 0 in the No column. prior_programming_experience becomes three columns: None, Some, and Strong. The model can then learn a separate weight for each category. That is different from coding None as 0, Some as 1, and Strong as 2, which would pretend those labels are evenly spaced numbers.

Why do we need to handle missing values? A blank cell cannot be multiplied by a weight, and it cannot become a 0/1 category. Nine students are missing study hours, twelve are missing sleep hours, and seven are missing prior programming experience. The numeric imputer fills a blank with the training-set median. The median is a better fill than the mean here because a few commutes are very long and would pull a mean upward. The category imputer fills a blank with the most common training category. Those fills are learned from the 200 training students, then applied to the test students. The word None is not one of those blanks. It is a real category, meaning the student had no prior programming experience. The load cell keeps it as a category so those 88 students are not filled in as if their answer were missing.

What does StandardScaler do? It rescales each numeric column so that, on the training set, the average is 0 and the standard deviation is 1. Study hours are small numbers, attendance is usually in the 70s to 100, and commute minutes can be near 0 or well over 100. Without scaling, a column with bigger numbers can push the model harder just because of its units. After scaling, a value of 1 means one standard deviation above that column's training average.

## Part 6 — Understand Data Scaling

Study hours, attendance, and commute time are all numbers, but they are not in the same units. To fix this, I used StandardScaler to compute z = (x - mu) / sigma for each numeric value in the set. In this new computation,  x is the original value, mu is the training-set mean, and sigma is the training-set standard deviation. After this step, each numeric training column has mean about 0 and standard deviation about 1. The next cell reads those training values back out of the fitted scaler and applies the formula to one student.

In [ ]:
numeric_pipeline_fitted = preprocessor.named_transformers_["numeric"]
scaler = numeric_pipeline_fitted.named_steps["scaler"]

training_mean = pd.Series(scaler.mean_, index=numeric_features).round(2)
training_std = pd.Series(scaler.scale_, index=numeric_features).round(2)

print("Training mean (mu):")
print(training_mean)
print()
print("Training standard deviation (sigma):")
print(training_std)

scaled_train = pd.DataFrame(
    X_train_prepared[:, : len(numeric_features)],
    columns=numeric_features,
)
scaled_summary = pd.DataFrame({
    "mean": scaled_train.mean(),
    "std": scaled_train.std(ddof=0),
    "min": scaled_train.min(),
    "max": scaled_train.max(),
})
print()
print("Scaled training columns:")
print(scaled_summary.round(2))

example_index = X_train["commute_minutes"].idxmax()
example = X_train.loc[
    example_index,
    ["study_hours_per_week", "attendance_rate", "commute_minutes"],
]
x = example["commute_minutes"]
mu = training_mean["commute_minutes"]
sigma = training_std["commute_minutes"]

print()
print("Example student, original values:")
print(example)
print(f"Commute z-score: ({x} - {mu}) / {sigma} = {round((x - mu) / sigma, 2)}")

scaled_test = pd.DataFrame(
    X_test_prepared[:, : len(numeric_features)],
    columns=numeric_features,
)
print()
print("Scaled test-set means, using the training mu and sigma:")
print(scaled_test.mean().round(3))

## Questions for Part 6

Why might scaling be useful for Logistic Regression? Logistic Regression learns a weight for each feature and adds those weighted features together. In the training set, study hours average about 7.4, attendance averages about 85, and commute time averages about 25 minutes. A one-hour change in study time and a one-point change in attendance are very different changes, but both are a change of 1 in the raw numbers. A column with larger numbers can then push the total harder just because of its units. After standardization, a change of 1 means one training standard deviation on every numeric feature, so the weights describe similar-sized changes.

Does standardization mean that all original values become numbers between 0 and 1? No. A standardized value says how many training standard deviations a student is from the training mean. Students below the mean get negative values, and students far above the mean get values larger than 1. In this training set the scaled numeric values run from about -3.2 to about 6.4. The longest training commute is 145 minutes. Its z-score is (145 - 24.86) / 18.77, which is about 6.4. Putting every value into the range 0 to 1 would be a different transformation.

Why should the test set not be used to calculate the mean and standard deviation used for scaling? Mu and sigma should describe the 200 training students. The scaler in this notebook was fit on that training set, and the same mu and sigma were then applied to the 50 test students. Their scaled means are close to 0, but not exactly 0, because the test students did not help calculate the average. If they had, the scale applied to a test student would depend on the other test students. The test score would then use information from the people we are trying to evaluate. A later student would also be scaled with the training mean and standard deviation, since that new student's value cannot be part of the original fit.

## Part 7 — Train a Logistic Regression Classifier

The next cell puts the preprocessing steps and a logistic regression classifier into one pipeline. Numeric columns are filled with the training median and standardized. Categorical columns are filled with the most common training category and one-hot encoded. Logistic regression then learns from those prepared columns. `.fit()` is called on the 200 training students only.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(transformers=[
        ("numeric", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ])),
    ("classifier", LogisticRegression()),
])

model.fit(X_train, y_train)

classifier = model.named_steps["classifier"]
prepared_columns = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.Series(classifier.coef_[0], index=prepared_columns).round(3)

print("Pipeline steps:", list(model.named_steps))
print("Classes:", list(classifier.classes_))
print("Training students used:", len(X_train))
print()
print("Intercept:", round(float(classifier.intercept_[0]), 3))
print()
print("Learned weight for each prepared column:")
print(coefficients)

## Questions for Part 7

What does .fit() do? It runs the pipeline from start to finish on the 200 training students. The numeric branch records each column's median, fills blanks, then records the mean and standard deviation and scales the column. The categorical branch records the most common category, fills blanks, and records the category names used for the 0/1 columns. Logistic regression then sees those prepared columns together with the Yes or No label and searches for an intercept and a weight for each prepared column.

What is learned from the training data? The preprocessing steps learn the fills and the scaling: the numeric medians, the most common prior-experience category, the training mean and standard deviation of each numeric column, and the category list None, Some, Strong, No, and Yes. The classifier learns how to turn the prepared columns into a Yes or No prediction. With this class order, the weights apply to the chance of Yes. The fitted intercept is about -0.52, and each prepared column gets its own weight. A larger positive weight pushes a student toward Yes. Those weights are estimated from the training students only.

Why is it useful to put preprocessing and the classifier into one pipeline? One call to .fit() learns the fills, the scaling, the encoding, and the weights in that order, and the classifier only sees the training students. Later, one predict call on a new student applies those stored steps before the classifier runs. The test students cannot slip into the medians or the mean and standard deviation, and a preprocessing step cannot be skipped or applied in a different order.

## Part 8 — Make Predictions

The fitted pipeline predicts high_exam_score for the 50 test students. predict() returns the class label and predict_proba() returns the probability of each class. The test students are still held out of training: these calls only apply the steps that were learned from the training set.

In [ ]:
predicted = model.predict(X_test)
yes_column = list(model.classes_).index("Yes")
probability_yes = model.predict_proba(X_test)[:, yes_column]

predictions = pd.DataFrame({
    "student_id": df.loc[X_test.index, "student_id"],
    "actual": y_test,
    "predicted": predicted,
})

print("Actual class and predicted class:")
print(predictions.head(10))

predictions["probability_yes"] = probability_yes.round(3)

print()
print("Actual class, predicted class, and probability of Yes:")
print(predictions.head(10))

distance_from_half = (predictions["probability_yes"] - 0.5).abs()
print()
print("Test students closest to probability 0.5:")
print(predictions.loc[distance_from_half.nsmallest(3).index])
print()
print("Test students with the highest probability of Yes:")
print(predictions.nlargest(3, "probability_yes"))

## Questions for Part 8

What is the difference between predict() and predict_proba()? predict() returns one label per student, Yes or No. It chooses the class with the higher probability, so a Yes probability above 0.5 becomes the label Yes, and a Yes probability below 0.5 becomes No. predict_proba() returns the probability of both classes. Those two probabilities add to 1. A student with a Yes probability of 0.820 has a No probability of 0.180, and predict() turns that pair into the single label Yes.

What does a predicted probability near 0.5 suggest? The model finds the two classes about equally likely, so the label is a close call. In this test set, one student has a Yes probability of 0.501 and is labeled Yes, while another has 0.496 and is labeled No. The evidence for those two students is nearly the same. The labels differ because 0.5 is the cutoff.

What does a probability near 0.95 suggest? The model assigns a strong chance to Yes. One test student has a Yes probability of 0.980, is predicted Yes, and actually is Yes. That student's habits line up with the high-score pattern learned from the training set. The number is still the model's estimated probability, so a confident prediction can be wrong for a different student.

## Part 9 — Evaluate the Classifier

The this cell scores the logistic regression predictions against those labels and compares the accuracy with the majority-class baseline from Part 4, which always predicted No and was right 60% of the time.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

predicted = model.predict(X_test)
labels = ["No", "Yes"]

accuracy = accuracy_score(y_test, predicted)
precision_yes = precision_score(y_test, predicted, pos_label="Yes")
recall_yes = recall_score(y_test, predicted, pos_label="Yes")
matrix = confusion_matrix(y_test, predicted, labels=labels)
matrix_table = pd.DataFrame(
    matrix,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)

correct = int((predicted == y_test).sum())
predicted_yes = int((predicted == "Yes").sum())
actual_yes = int((y_test == "Yes").sum())
true_yes = int(matrix_table.loc["Actual Yes", "Predicted Yes"])

print("Correct test predictions:", correct, "of", len(y_test))
print("Logistic regression accuracy:", round(accuracy, 3))
print("Majority-class baseline accuracy:", round(baseline_accuracy, 3))
print()
print("Confusion matrix:")
print(matrix_table)
print()
print("Precision for Yes:", true_yes, "of", predicted_yes, "=", round(precision_yes, 3))
print("Recall for Yes:", true_yes, "of", actual_yes, "=", round(recall_yes, 3))

## Questions for Part 9

Accuracy is the fraction of all test predictions that were correct. The model was right on 42 of the 50 test students, which is 84%. The majority-class baseline from Part 4 always predicted No and was right on 30 of 50, which is 60%. Logistic regression is right on 12 more test students than that simple guess, so its accuracy is above the baseline. The other counts show what that 84% is made of.

The confusion matrix splits the 50 students into four groups. 28 were actually No and predicted No. 14 were actually Yes and predicted Yes. Those 42 are the correct predictions. 2 were actually No but predicted Yes. 6 were actually Yes but predicted No.

Precision for Yes asks: among students predicted to have a high exam score, what fraction actually did? The model predicted Yes for 16 test students, and 14 of them really were Yes. Precision is 14 / 16, or 87.5%. When this model says Yes, it is right most of the time. The other 2 predicted high scores were students who scored below 80.

Recall for Yes asks: among students who actually had a high exam score, what fraction did the model identify? 20 test students were actually Yes, and the model found 14 of them. Recall is 14 / 20, or 70%. The model missed 6 students who did score 80 or above and labeled them No. An accuracy of 84% can sit next to a recall of 70%, because the missed high scores are only part of the 50 predictions.

## Part 10 — Understand the Confusion Matrix

Yes is the positive class, because the question is whether the student has a high exam score. No is the negative class. The next cell displays the same test-set matrix as Part 9, with each cell named.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

matrix = confusion_matrix(y_test, predicted, labels=["No", "Yes"])
true_negative = int(matrix[0, 0])
false_positive = int(matrix[0, 1])
false_negative = int(matrix[1, 0])
true_positive = int(matrix[1, 1])

named_matrix = pd.DataFrame(
    [
        [f"True negative: {true_negative}", f"False positive: {false_positive}"],
        [f"False negative: {false_negative}", f"True positive: {true_positive}"],
    ],
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)
print(named_matrix)

ConfusionMatrixDisplay(matrix, display_labels=["No", "Yes"]).plot()
plt.title("Confusion Matrix for high_exam_score")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.show()

print("True negatives:", true_negative)
print("False positives:", false_positive)
print("False negatives:", false_negative)
print("True positives:", true_positive)

## Questions for Part 10

How many false positives did the model produce? 2. These are test students whose actual class is No and whose predicted class is Yes.

How many false negatives? 6. These are test students whose actual class is Yes and whose predicted class is No.

The other two cells are correct predictions. A true negative is an actual No that the model also called No: 28 students. A true positive is an actual Yes that the model also called Yes: 14 students.

A false positive, in this problem, means the model predicts that the student will have a high exam score, and the student scores below 80. The warning is too optimistic. Two test students were labeled Yes even though their exam score was under the cutoff.

A false negative means the model predicts that the student will not have a high exam score, and the student scores 80 or above. The model misses a student who did land in the high group. Six test students were labeled No even though their exam score was at least 80. That is the same group of six students counted in the recall from Part 9: the model found 14 of the 20 actual Yes students and missed these 6.

## Part 11 — Inspect Individual Errors

The classifier missed 8 of the 50 test students: 2 false positives and 6 false negatives. The tables below show the habits the model actually used, plus the exam score that defined the label. The exam score was not a predictor. A pattern in these rows is a hypothesis about the model, not a claim that a habit caused an exam result.

In [ ]:
error_columns = [
    "student_id",
    "exam_score",
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "prior_programming_experience",
    "study_group",
    "commute_minutes",
]

yes_column = list(model.classes_).index("Yes")
probability_yes = model.predict_proba(X_test)[:, yes_column]

errors = df.loc[X_test.index, error_columns].copy()
errors["actual"] = y_test
errors["predicted"] = predicted
errors["probability_yes"] = probability_yes.round(3)
errors = errors[errors["actual"] != errors["predicted"]]

false_positives = errors[
    (errors["actual"] == "No") & (errors["predicted"] == "Yes")
].sort_values("probability_yes", ascending=False)

false_negatives = errors[
    (errors["actual"] == "Yes") & (errors["predicted"] == "No")
].sort_values("probability_yes", ascending=False)

print("False positives:", len(false_positives))
print(false_positives.to_string(index=False))
print()
print("False negatives:", len(false_negatives))
print(false_negatives.to_string(index=False))
print()
print("Training medians by actual class:")
training_view = X_train.copy()
training_view["high_exam_score"] = y_train
print(training_view.groupby("high_exam_score")[numeric_features].median().round(1))

## Questions for Part 11

Do you notice anything interesting about the mistakes? All eight wrong predictions are uncertain. Their Yes probabilities run from 0.299 to 0.541. The model was not strongly sure about any student it got wrong. Two of the mistakes also sit almost on the exam cutoff that creates the label. S170 scored 79.6 and was predicted Yes, with probability 0.541. S038 scored 80.1 and was predicted No, with probability 0.338. Those exam scores are on opposite sides of 80 by less than half a point, so the labels are opposite even though both scores are essentially at the boundary. However, S133 is a different kind of false positive. The Yes probability is 0.501, so the predicted label is Yes only because it is just above 0.5, but the exam score is 66.8. The habits the model could see include 9.7 study hours and 92.9% attendance. In the training set, the median study hours for Yes students were about 8.9, and the median attendance for Yes students was about 89.5. Those two values sit on the higher side of the training Yes group, while the exam score is well below 80.

Can you form a hypothesis about why the model may have found some of these students difficult? My hypothesis is that a hard cutoff at 80 makes students near that score difficult. Similar habits can belong to opposite labels when one exam is just under 80 and another is just over it. A probability near 0.5 is a plausible place for that kind of mistake. S170 fits that pattern: high attendance, study hours a bit above the Yes median, and an exam score 0.4 points under the cutoff.

## Part 12 — Target Leakage Experiment

This is a deliberately bad experiment. exam_score is added to the predictors, and logistic regression is trained again on the same 200 training students. The test students stay the same, so the accuracy can be compared with the legitimate model from Part 9.

In [ ]:
leaky_numeric_features = numeric_features + ["exam_score"]
leaky_columns = leaky_numeric_features + categorical_features

X_train_leaky = df.loc[X_train.index, leaky_columns]
X_test_leaky = df.loc[X_test.index, leaky_columns]

leaky_model = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(transformers=[
        ("numeric", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), leaky_numeric_features),
        ("categorical", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ])),
    ("classifier", LogisticRegression()),
])

leaky_model.fit(X_train_leaky, y_train)
leaky_predicted = leaky_model.predict(X_test_leaky)
leaky_accuracy = accuracy_score(y_test, leaky_predicted)
leaky_matrix = confusion_matrix(y_test, leaky_predicted, labels=["No", "Yes"])

leaky_names = leaky_model.named_steps["preprocessor"].get_feature_names_out()
leaky_coefficients = pd.Series(
    leaky_model.named_steps["classifier"].coef_[0],
    index=leaky_names,
).round(3)

print("Legitimate model accuracy:", round(accuracy, 3))
print("Leaky model accuracy:", round(leaky_accuracy, 3))
print("Leaky model correct:", int((leaky_predicted == y_test).sum()), "of", len(y_test))
print()
print("Leaky confusion matrix:")
print(pd.DataFrame(
    leaky_matrix,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
))
print()
print("Largest learned weights:")
print(leaky_coefficients.reindex(leaky_coefficients.abs().sort_values(ascending=False).index).head(5))

## Questions for Part 12

What happened to the accuracy? On the same 50 test students, accuracy rose from 84% to 100%. The legitimate model was right on 42 students. The model that can see exam_score is right on all 50: all 30 actual No students and all 20 actual Yes students. The learned weight on exam_score is about 4.6, and the habit weights shrink to a few tenths.

Why did this happen? high_exam_score was defined as Yes when exam_score is 80 or above and No otherwise. Once exam_score is an input, the label is a rule applied to one of the columns. The model can place a cutoff at 80 and recover the class without needing study hours, sleep, attendance, or the other habits. This is target leakage: the feature contains the answer the model is supposed to predict.

Why is this model invalid even though its accuracy is 100%? The score would not be available for a student who has not taken the exam yet. That is the situation the legitimate model is for. A perfect test score here shows that the pipeline reconstructed the labeling rule. It does not show that the habit columns can identify who will score 80 or above. Reporting 100% as the performance of a habit model would describe a different, already-answered question.

## Part 13 — A Legitimate Feature Experiment

exam_score is removed again. This part asks whether quiz_average helps when it is added to the original habit columns. The training students and the test students stay the same as in Part 3.

My prediction: I expect adding quiz average to raise accuracy, precision, and recall by a modest amount. In the previous we did, quiz average and exam score moved together more clearly than study hours and exam score did. Students with higher quiz averages tended to have higher exam scores. Quiz average is a different column from exam_score, so I do not expect the perfect accuracy from the leakage experiment. I expect a small gain over the habit-only model.

In [ ]:
quiz_numeric_features = numeric_features + ["quiz_average"]
quiz_columns = quiz_numeric_features + categorical_features

X_train_quiz = df.loc[X_train.index, quiz_columns]
X_test_quiz = df.loc[X_test.index, quiz_columns]

quiz_model = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(transformers=[
        ("numeric", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), quiz_numeric_features),
        ("categorical", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ])),
    ("classifier", LogisticRegression()),
])

quiz_model.fit(X_train_quiz, y_train)
quiz_predicted = quiz_model.predict(X_test_quiz)

quiz_accuracy = accuracy_score(y_test, quiz_predicted)
quiz_precision = precision_score(y_test, quiz_predicted, pos_label="Yes")
quiz_recall = recall_score(y_test, quiz_predicted, pos_label="Yes")
quiz_matrix = confusion_matrix(y_test, quiz_predicted, labels=["No", "Yes"])

comparison = pd.DataFrame(
    {
        "original": [accuracy, precision_yes, recall_yes],
        "with_quiz_average": [quiz_accuracy, quiz_precision, quiz_recall],
    },
    index=["accuracy", "precision_yes", "recall_yes"],
).round(3)

print(comparison)
print()
print("Quiz model correct:", int((quiz_predicted == y_test).sum()), "of", len(y_test))
print("Confusion matrix with quiz_average:")
print(pd.DataFrame(
    quiz_matrix,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
))

## Questions for Part 13

Did quiz_average improve the model? Yes, by one test student. The habit-only model was right on 42 of 50. The model with quiz average is right on 43 of 50. The confusion matrix still has 2 false positives. False negatives fall from 6 to 5, so the extra correct prediction is one actual Yes student the original model missed.

Which metrics changed? All three rose but only slightly.
- Accuracy: 84% to 86% (42 of 50 to 43 of 50).
- Precision for Yes: 87.5% to 88.2% (14 of 16 predicted Yes students, to 15 of 17).
- Recall for Yes: 70% to 75% (14 of 20 actual Yes students, to 15 of 20).

Was the original prediction correct? The direction and the size were both right. I expected a modest gain, not the perfect accuracy from adding exam_score. That is what happened. Quiz average is associated with the exam score, and the test metrics moved up by one student. It did not reconstruct the label the way exam_score did.

Is quiz_average available when the prediction is supposed to be made? It depends on the moment of the prediction. Quizzes usually happen before the final exam, so a prediction made after the quizzes and before the exam can use quiz_average. In that setting it is a real earlier measurement, not the exam score, and it is not the leakage from Part 12. A prediction made at the start of the term, from habits alone, cannot use it, because those quiz scores do not exist yet. The original model matches that earlier decision, but the 86% model matches a later one.

## Part 14 — My Own Experiment: What Does Scaling Change?

The original model and this comparison use the same seven habit columns, the same 200 training students, and the same 50 test students. The only change is that the numeric branch fills missing values and then skips StandardScaler.

My prediction: I think that removing StandardScaler will lower test accuracy, and that the numeric weights will become hard to compare. Study hours, attendance, and commute time use different units. Logistic regression adds each weight times its feature, so a change of 1 means one hour for study time and one percentage point for attendance. I also think the solver will need more iterations, because the columns will no longer share a scale.

In [ ]:
unscaled_model = Pipeline(steps=[
    ("preprocessor", ColumnTransformer(transformers=[
        ("numeric", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_features),
        ("categorical", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ])),
    ("classifier", LogisticRegression(max_iter=1000)),
])

unscaled_model.fit(X_train, y_train)
unscaled_predicted = unscaled_model.predict(X_test)

unscaled_accuracy = accuracy_score(y_test, unscaled_predicted)
unscaled_precision = precision_score(y_test, unscaled_predicted, pos_label="Yes")
unscaled_recall = recall_score(y_test, unscaled_predicted, pos_label="Yes")

metric_comparison = pd.DataFrame(
    {
        "with_scaling": [accuracy, precision_yes, recall_yes],
        "without_scaling": [unscaled_accuracy, unscaled_precision, unscaled_recall],
    },
    index=["accuracy", "precision_yes", "recall_yes"],
).round(3)
print(metric_comparison)
print()
print("Iterations with scaling:", int(model.named_steps["classifier"].n_iter_[0]))
print("Iterations without scaling:", int(unscaled_model.named_steps["classifier"].n_iter_[0]))

scaled_weights = pd.Series(
    model.named_steps["classifier"].coef_[0],
    index=model.named_steps["preprocessor"].get_feature_names_out(),
)
unscaled_weights = pd.Series(
    unscaled_model.named_steps["classifier"].coef_[0],
    index=unscaled_model.named_steps["preprocessor"].get_feature_names_out(),
)
training_std = model.named_steps["preprocessor"].named_transformers_["numeric"].named_steps["scaler"].scale_
numeric_names = [f"numeric__{column}" for column in numeric_features]
weight_comparison = pd.DataFrame({
    "scaled_weight": scaled_weights[numeric_names],
    "unscaled_weight": unscaled_weights[numeric_names],
    "unscaled_weight_times_std": unscaled_weights[numeric_names].to_numpy() * training_std,
}).round(3)
print()
print("Numeric weights:")
print(weight_comparison)

disagreement = predicted != unscaled_predicted
print()
print("Test students whose predicted class changed:", int(disagreement.sum()))
print(df.loc[X_test.index[disagreement], ["student_id", "exam_score"]].assign(
    scaled_prediction=predicted[disagreement],
    unscaled_prediction=unscaled_predicted[disagreement],
    actual=y_test.loc[X_test.index[disagreement]].to_numpy(),
))

## My Experiment Results

The accuracy part of my hypothesis was not supported, but removing StandardScaler raised the test scores by one student.
- Accuracy: 84% to 86% (42 of 50 to 43 of 50).
- Precision for Yes: 87.5% to 88.2% (14 of 16 to 15 of 17).
- Recall for Yes: 70% to 75% (14 of 20 to 15 of 20).

The two models disagree on one test student, S081. The scaled model predicted No. The unscaled model predicted Yes, and the actual class is Yes. In Part 11, S081 was a boundary case: 5.6 study hours, 36 practice problems, an exam score of 87.9, and a scaled Yes probability of 0.496. One student on a 50-student test set is a small change. It does not show that leaving the features unscaled is a better model.

The rest of the hypothesis was supported. The unscaled numeric weights are in raw units: about 0.34 per study hour, 0.047 per attendance point, and 0.004 per commute minute. Those numbers look as if attendance and commute barely matter. Multiplying each unscaled weight by that column's training standard deviation brings them back near the scaled weights. Study hours are about 0.96 versus 0.92. Attendance is about 0.45 versus 0.43. The associations are similar. The raw weights are hard to compare because the units differ. The 0/1 category weights barely moved, which fits because those columns were already on a small scale.

The solver also needed more work. The scaled model finished in 14 iterations. The unscaled model needed 326. The default limit is 100 iterations, so an unscaled fit with the usual settings stops early and warns that it has not converged. Scaling did not create the pattern in the habits. It put the numeric columns on a shared scale, so the weights can be compared and the fit can finish.

## Part 15 — AI Reflection

The AI wrote the scikit-learn code for this notebook(the stratified split, the preprocessing pipeline, the logistic regression model, the metrics, and the later experiments). The pipeline suggestion was to fill numeric blanks with the training median, scale those columns with StandardScaler, fill category blanks with the most common training category, and one-hot encode them. One-hot encoding was explained as a separate 0/1 column for each category, so None, Some, and Strong are not treated as the numbers 0, 1, and 2. StandardScaler was explained as z = (x - mu) / sigma, fit on the training students only. Logistic regression was explained as learning an intercept and a weight for each prepared column. The confusion matrix was interpreted as 28 true negatives, 2 false positives, 6 false negatives, and 14 true positives.

The first was a debugging suggestion in the load cell. A normal read_csv was going to treat the word None in prior_programming_experience as a missing value. The AI said that word is a real category, meaning no prior programming experience, and that only blank cells should be missing. I then accepted that change, and keep_default_na=False and na_values=[""]. I verified it by counting the raw file. The column contains Some 107 times, None 88 times, Strong 48 times, and 7 blanks. After the fix, the notebook reports 7 missing values in that column, not 95. If I had skipped the check, 88 students would have been filled in as if their answer were blank.

The second was the experiment in Part 14. I asked to study scaling. The AI's hypothesis was: removing StandardScaler will lower test accuracy, the numeric weights will become hard to compare, and the solver will need more iterations, because study hours, attendance, and commute time use different units. I accepted the experiment design and ran it on the same split, how the accuracy part was not supported. Accuracy went from 84% to 86%, precision from 87.5% to 88.2%, and recall from 70% to 75%. The models disagreed on one test student, S081, whose scaled Yes probability was 0.496. The weight part and the iteration part were supported. Unscaled weights are about 0.34 per study hour and 0.004 per commute minute, and multiplying by the training standard deviation brings them back near the scaled weights. The scaled fit finished in 14 iterations. The unscaled fit needed 326, past the default limit of 100.

The assumption I needed to question was that a shared scale would raise accuracy. Part 6 gives a good reason to scale, so that weights are comparable and the fit can finish. It does not follow that the test accuracy must fall when the scaler is removed. On this 50-student test set, the change was one boundary student. I also had to question the imputation choice. Median, rather than mean, was justified by the long commute tail from our previous assigmnet. That is a reasonable choice, and this notebook does not show what the predictions would be if the fill were the mean instead.

What I learned is that the AI is useful for assembling the pipeline and for naming the cells of a confusion matrix, and that a fluent explanation is not the same as a result. The leakage model at 100% and the habit model at 84% were both produced by the same kind of code. The difference was whether exam_score was allowed into the features. I had to check the code against the file, the class counts, and the test metrics before treating a suggestion as something the data had shown.